# 09 — Enhancement Round 3 (remaining plan backlog)

Round 3 executes the rest of the planners' backlog: Fare-bin replacement, Cabin Known-only, targeted RF regularization, and triple-grouping model choice. Same pre-committed gates, same hard rules. No test labels, no downloads, no leaderboard use.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))


In [ ]:
R = json.loads((ROOT/'reports'/'experiment_results.json').read_text())
A = json.loads((ROOT/'reports/gap4_r3a.json').read_text())
B = json.loads((ROOT/'reports/gap4_r3b.json').read_text())
C = json.loads((ROOT/'reports/gap4_r3c.json').read_text())
D = json.loads((ROOT/'reports/gap4_r3d.json').read_text())
RF = R['best_cv']
print('reference RF:', round(RF, 4))

## R3A Fare bins + R3B Known-only: both REJECT
Bins hurt both CVs and every fare slice (Q2 stays ~0.80); dropping Deck hurts random and collapses the known-cabin slice it was meant to fix — deck carried real signal.


In [ ]:
assert abs(A['results']['base_random']['mean'] - RF) < 1e-9
assert abs(B['results']['base_random']['mean'] - RF) < 1e-9
assert A['verdict'] == 'REJECT' and B['verdict'] == 'REJECT'
print('R3A variant: random %.4f group %.4f' % (A['results']['variant_random']['mean'], A['results']['variant_group']['mean']))
print('R3B variant: random %.4f group %.4f' % (B['results']['variant_random']['mean'], B['results']['variant_group']['mean']))
assert A['results']['variant_random']['mean'] < RF and A['results']['variant_group']['mean'] < 0.7991
assert B['results']['variant_random']['mean'] < RF
print('R3A+R3B confirmed: representation cuts go the wrong way on both validations')

## R3C regularization: closest call still rejects (t = 1.98 < 2.0)
R2/R4 gain +0.0169 group with acceptable random cost and better Brier — yet paired-t 1.9763 misses the pre-committed 2.0 by 0.024. Gates are gates: REJECT. Falsifier fires for structure-not-depth.


In [ ]:
assert C['verdict'].startswith('REJECT')
r2 = C['variants']['R2']
print('R2: group %.4f (delta %+.4f) random %.4f brier %.4f paired-t %.2f' % (
    r2['group']['mean'], r2['delta_group'], r2['random']['mean'], r2['oof_brier_random'], r2['paired_t_vs_R0_group']['t']))
assert abs(r2['paired_t_vs_R0_group']['t'] - 1.9763) < 5e-3, 't-gate miss not reproduced'
print('R3C confirmed: nearest miss in the program still fails the t-gate')

## R3D triple-grouping: LR sweeps groups, still RETAIN RF
LR wins ticket, surname, and famticket (pooled-t +3.16) — yet fails the random gate (−0.0157 vs −0.0067 tolerance) with Brier unmeasured. Note: RF-famticket here (0.8025) differs from the prior 0.8137 purely by separator-driven shuffle assignment (`|` vs `__`) — grouping CV itself is shuffle-sensitive, another reason not to crown winners on it.


In [ ]:
assert D['verdict'] == 'RETAIN'
w = D['per_grouping_winners']
print('winners:', {k: (v if isinstance(v, str) else v.get('winner', v)) for k, v in w.items()})
print('R3D confirmed: grouping-dependent robustness is not a switch signal')

## Round-3 outcome: RF retained a third time; plan backlog exhausted

- All four remaining planner tracks executed with pre-committed gates; all reject/retain.
- Nearest miss (R3C R2/R4, t=1.98) shows the gates bite exactly where they should: promising but inseparable.
- Production model and submission unchanged. Recorded as E18–E21.
- Standing lesson unchanged and strengthened: only deltas clearly outside the ~0.013 seed range merit adoption. Nothing tested so far qualifies.
